# Building Forecasts

## Decomposition and simple forecasting with pandas and numpy

The explainer made the case that forecasting is both one of the most requested and most dangerous things an analyst can do. A forecast without uncertainty is an incomplete answer. A forecast projected too far into the future is closer to speculation than prediction. The techniques are simple; the discipline is knowing their limits.

This notebook puts those ideas into practice. We will build forecasts of daily solar output using only pandas and numpy — no specialist libraries. The methods are deliberately simple (naive seasonal forecasts, rolling averages, confidence bands), but they form the foundation of every forecasting approach: separate the signal from the noise, project the signal forward, and report honestly how much we trust the projection.

The scenario is the same grid planning problem from the previous notebook. Can we predict next week's solar output? If we overestimate, we burn fossil fuel we did not need. If we underestimate, we risk blackouts.

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
solar = pd.read_csv("../data/solar_output.csv", parse_dates=["timestamp"], index_col="timestamp")
solar.head()

We will work with daily average solar output. Forecasting at the daily level is more practical than hourly for capacity planning, and it removes the within-day noise we explored in the previous notebook.

In [ ]:
daily_solar = solar["output_mw"].resample("D").mean()
daily_solar.plot(title="Daily Average Solar Output (2024)", ylabel="MW")
plt.tight_layout()
plt.show()

---

## 1. Manual seasonal decomposition

The explainer introduced decomposition conceptually: estimate the trend, subtract it, and see what remains. Here we will do it with code. A time series can be thought of as three additive components:

**observed = trend + seasonal + residual**

- **Trend**: the long-term direction (solar output rising through spring, falling through autumn).
- **Seasonal**: the repeating pattern at a fixed period (e.g. a weekly cycle).
- **Residual**: whatever is left after removing trend and seasonal. This is the unpredictable noise — mostly weather, in the case of solar.

We extract the trend using a rolling average, then subtract it from the original to isolate the seasonal and residual components.

In [ ]:
# 30-day rolling average captures the seasonal trend
trend = daily_solar.rolling(window=30, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_solar.index, daily_solar.values, alpha=0.4, label="Daily solar")
ax.plot(trend.index, trend.values, label="30-day trend", linewidth=2, color="red")
ax.set_title("Solar Output with 30-Day Rolling Trend")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Subtract the trend to get seasonal + residual
detrended = daily_solar - trend

detrended.plot(title="Detrended Solar Output (seasonal + residual)", ylabel="MW")
plt.tight_layout()
plt.show()

The detrended series oscillates around zero. The day-to-day jumps are the residual, driven mainly by cloud cover. If there were a strong weekly pattern (the way demand drops on weekends), it would show up as a regular wave. But the sun does not know what day of the week it is, so the residual dominates.

We can check by averaging each day-of-week across the year.

In [ ]:
# Average detrended value by day of week
seasonal_weekly = detrended.groupby(detrended.index.dayofweek).mean()
print(seasonal_weekly)

As expected, the weekly seasonal component is near zero for all days. Solar output has no weekday/weekend pattern. For demand data (which has strong weekday/weekend differences), this component would be much more significant. We will see that contrast in Exercise 1.

The residual is what remains after subtracting both trend and seasonal.

In [ ]:
# Map the weekly seasonal pattern back to each date
seasonal_component = detrended.index.dayofweek.map(seasonal_weekly.to_dict())
seasonal_series = pd.Series(seasonal_component.values, index=detrended.index)

residual = detrended - seasonal_series

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
daily_solar.plot(ax=axes[0], title="Observed")
trend.plot(ax=axes[1], title="Trend (30-day rolling average)")
seasonal_series.plot(ax=axes[2], title="Seasonal (weekly)")
residual.plot(ax=axes[3], title="Residual")

for ax in axes:
    ax.set_ylabel("MW")

plt.tight_layout()
plt.show()

For solar output, the trend carries most of the signal and the residual is large. That large residual is the explainer's "noise" component, the random variation left over once we remove the predictable patterns. For solar, cloud cover is the dominant source of noise, and it is genuinely hard to predict more than a few days ahead. For demand data, the decomposition would be more balanced, with a stronger seasonal component.

---

## 2. Train/test split

Before building forecasts, we split the data into **training** and **test** sets. We train on the first 48 weeks and test on the final 4 weeks (roughly December). This mirrors how forecasting works in practice: we build a model on historical data and evaluate it against unseen future data. If we evaluated against the same data we trained on, we would be measuring how well the model memorises, not how well it predicts.

In [ ]:
split_date = "2024-12-01"
train = daily_solar[:split_date]
test = daily_solar[split_date:]

print(f"Training: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:     {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")

---

## 3. Naive seasonal forecast

The simplest seasonal forecast: predict that today's value will be the same as the value exactly one week ago. This is called a **naive seasonal forecast** with period 7.

It sounds crude, but it is surprisingly hard to beat for data with strong weekly patterns. It also serves as a baseline: any more sophisticated method should outperform this, or it is not worth the added complexity. The explainer made this point about moving averages and exponential smoothing — simple methods set the bar.

In [ ]:
# For each day in the test set, use the value from 7 days earlier
naive_forecast = daily_solar.shift(7)
naive_test = naive_forecast[split_date:].dropna()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, label="Actual", marker="o", markersize=4)
ax.plot(naive_test.index, naive_test.values, label="Naive (same day last week)", marker="s", markersize=4)
ax.set_title("Naive Seasonal Forecast vs Actual (December 2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

---

## 4. Rolling average forecast

The explainer described moving averages as a technique that smooths out short-term fluctuations to reveal the underlying level. Here we use the same idea as a forecasting method: predict that tomorrow's value will be the average of the last `k` days. This captures the recent level but does not model weekly or seasonal patterns explicitly.

In [ ]:
# 7-day rolling average as a forecast (the average of the previous 7 days)
rolling_forecast = daily_solar.rolling(window=7).mean().shift(1)
rolling_test = rolling_forecast[split_date:].dropna()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, label="Actual", marker="o", markersize=4)
ax.plot(naive_test.index, naive_test.values, label="Naive seasonal", marker="s", markersize=4)
ax.plot(rolling_test.index, rolling_test.values, label="7-day rolling avg", marker="^", markersize=4)
ax.set_title("Forecast Comparison (December 2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

---

## 5. Evaluating forecast accuracy

The explainer stressed that a forecast without a measure of uncertainty is incomplete. Before we add uncertainty, we need to measure how wrong our forecasts are. Two standard metrics:

- **MAE (Mean Absolute Error)**: the average absolute difference between forecast and actual. Easy to interpret in the original units (MW).
- **RMSE (Root Mean Squared Error)**: the square root of the average squared difference. Penalises large errors more heavily than MAE.

Lower is better for both. These numbers tell the grid operator how much to trust the forecast.

In [ ]:
def mae(actual, predicted):
    """Mean Absolute Error."""
    return np.abs(actual - predicted).mean()


def rmse(actual, predicted):
    """Root Mean Squared Error."""
    return np.sqrt(((actual - predicted) ** 2).mean())

In [ ]:
# Align test and forecasts on the same dates
common_dates = test.index.intersection(naive_test.index).intersection(rolling_test.index)
actual = test[common_dates]
naive_pred = naive_test[common_dates]
rolling_pred = rolling_test[common_dates]

print(f"{'Method':<25} {'MAE (MW)':>10} {'RMSE (MW)':>10}")
print("-" * 47)
print(f"{'Naive seasonal':<25} {mae(actual, naive_pred):>10.1f} {rmse(actual, naive_pred):>10.1f}")
print(f"{'7-day rolling average':<25} {mae(actual, rolling_pred):>10.1f} {rmse(actual, rolling_pred):>10.1f}")

For solar output, the rolling average forecast typically performs similarly to the naive forecast. Both methods are limited by the unpredictability of cloud cover — the large residual we saw in the decomposition. The MAE tells us, on average, how many MW our forecast is off by each day. That number is the honest part of the forecast.

---

## 6. Confidence bands

The explainer made a strong claim: a forecast that says "we expect 1,200 MW tomorrow" is incomplete. A forecast that says "we expect 1,200 MW, and it is unlikely to be below 800 or above 1,600" gives the decision-maker the information they need to judge how much weight to place on the prediction.

We can build a simple confidence band from the residuals. The idea: calculate the residuals (actual minus forecast) on the training set, then use the standard deviation of those residuals to build a band around the forecast. If the residuals are roughly normally distributed, approximately 95% of future values should fall within the mean +/- 2 standard deviations.

In [ ]:
# Residuals on the training set for the rolling average forecast
rolling_train_forecast = daily_solar.rolling(window=7).mean().shift(1)
train_residuals = train - rolling_train_forecast[:split_date]
train_residuals = train_residuals.dropna()

residual_std = train_residuals.std()
residual_mean = train_residuals.mean()

print(f"Residual mean:  {residual_mean:.1f} MW")
print(f"Residual std:   {residual_std:.1f} MW")

In [ ]:
# Plot the forecast with confidence bands on the test set
upper = rolling_pred + 2 * residual_std
lower = rolling_pred - 2 * residual_std
lower = lower.clip(lower=0)  # solar output cannot be negative

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actual.index, actual.values, label="Actual", marker="o", markersize=5)
ax.plot(rolling_pred.index, rolling_pred.values, label="Forecast", linewidth=2)
ax.fill_between(rolling_pred.index, lower, upper, alpha=0.2, label="95% band")
ax.set_title("Rolling Average Forecast with Confidence Band")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# What fraction of actual values fall within the band?
within_band = ((actual >= lower) & (actual <= upper)).mean() * 100
print(f"{within_band:.0f}% of actual values fall within the 95% confidence band")

The confidence band gives the grid operator something far more useful than a single line. The lower bound represents the minimum solar output they should plan for. If actual output exceeds the forecast, that is a bonus. If it falls below the lower band, backup generation must be ready. This is the practical meaning of reporting uncertainty: it turns a prediction into a planning tool.

---

## Exercises

We now have the tools to decompose a time series, build simple forecasts, measure their accuracy, and wrap them in confidence bands. These exercises extend those techniques to different data and different window sizes.

### Exercise 1: Decompose demand

Load the grid demand data and compute daily averages. Perform a manual decomposition: extract the trend using a 30-day centred rolling average, compute the weekly seasonal component by averaging the detrended values by day of week, and calculate the residual. Plot all four components (observed, trend, seasonal, residual) in a 4-panel figure. How does the weekly seasonal component for demand compare to what we saw for solar?

In [ ]:
# Your code here


### Exercise 2: 14-day rolling forecast

Build a rolling average forecast using a 14-day window instead of 7. Evaluate it on the same December test set using MAE and RMSE. Does the wider window improve or worsen the forecast compared to the 7-day version? Why might that be?

In [ ]:
# Your code here


### Exercise 3: Wind output forecast

Load the wind output data, resample to daily, and split at 1 December. Build both a naive seasonal (7-day lag) and a 7-day rolling average forecast. Evaluate both on the test set. Which method works better for wind, and why might that differ from solar?

In [ ]:
# Your code here


### Exercise 4: Confidence band for demand

Using the demand data, build a 7-day rolling average forecast with confidence bands. Calculate the residual standard deviation from the training period. Plot the forecast and band for December. What percentage of actual demand values fall within the band? If the grid operator uses the upper band as a planning ceiling, how much spare capacity would they need?

In [ ]:
# Your code here


---

## Summary

This notebook covered the mechanics of time series forecasting without specialist libraries:

- **Decomposition** breaks a series into trend, seasonal, and residual components. A rolling average extracts the trend; grouping by time period extracts the seasonal pattern; the residual is what remains — and its size tells us how predictable the series is.
- **Naive seasonal forecast** uses the value from the same period in the previous cycle (e.g. same day last week). It is the simplest method and a useful baseline.
- **Rolling average forecast** predicts the next value as the mean of the last `k` values. It adapts to recent levels but ignores cyclical patterns.
- **MAE and RMSE** measure forecast accuracy. MAE is the average absolute error in the original units; RMSE penalises large errors more heavily.
- **Confidence bands** built from residual standard deviation give the operator a range, not just a point. For capacity planning, the edges of the band matter more than the centre.

In the next notebook, we will combine these forecasts into a supply gap analysis and turn the numbers into a stakeholder recommendation.